In [0]:
# Mount ADLS Gen2

tiers = ["bronze","silver","gold"]
adls_paths = {tier: f"abfss://{tier}@databricksstore88.dfs.core.windows.net/" for tier in tiers}

# Accessing ADLS paths
bronze_adls = adls_paths["bronze"]
silver_adls = adls_paths["silver"]
gold_adls = adls_paths["gold"]

dbutils.fs.ls(bronze_adls)
dbutils.fs.ls(silver_adls)
dbutils.fs.ls(gold_adls)

[]

In [0]:
import requests
import json
from datetime import date,timedelta 

In [0]:
start_date = date.today() - timedelta(1)
end_date = date.today()

In [0]:
# construct API URL with start date and end date, formatted for json output.

url = f"https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime={start_date}&endtime={end_date}"

try:
    # Request to fetch data
    response = requests.get(url)

    #check if request was successful
    response.raise_for_status() #Raise HTTPError for bas responses
    #return value of features only. If features doesn't exist, give me an empty list []." JSON string → Python object
    data = response.json().get('features',[])

    if not data:
        print("No data returned for specified data range")
    else:
        #specify ADLS path
        file_path = f"{bronze_adls}/{start_date}_earthquake_data.json"  

        #save the JSON data , Python object → JSON string , 
        #indent =4 : Format the JSON using 4 spaces for indentation so that it is human-readable.
         
        json_data = json.dumps(data,indent=4)
        dbutils.fs.put(file_path,json_data,overwrite=True)
        print(f"Data successfully saved to {file_path}")
except requests.exceptions.RequestException as e:
    print("Error while fetching datat from API: {e}")        

Wrote 282231 bytes.
Data successfully saved to abfss://bronze@databricksstore88.dfs.core.windows.net//2026-09-15_earthquake_data.json


In [0]:
data[0]

{'type': 'Feature',
 'properties': {'mag': 1.18,
  'place': '4 km W of Mammoth Lakes, CA',
  'time': 1789516675120,
  'updated': 1789526541546,
  'tz': None,
  'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/nc75436297',
  'detail': 'https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=nc75436297&format=geojson',
  'felt': None,
  'cdi': None,
  'mmi': None,
  'alert': None,
  'status': 'automatic',
  'tsunami': 0,
  'sig': 21,
  'net': 'nc',
  'code': '75436297',
  'ids': ',nc75436297,',
  'sources': ',nc,',
  'types': ',nearby-cities,origin,phase-data,scitech-link,',
  'nst': 10,
  'dmin': 0.002426,
  'rms': 0.02,
  'gap': 119,
  'magType': 'md',
  'type': 'earthquake',
  'title': 'M 1.2 - 4 km W of Mammoth Lakes, CA'},
 'geometry': {'type': 'Point',
  'coordinates': [-119.029167175293, 37.6404991149902, 0.129999995231628]},
 'id': 'nc75436297'}

In [0]:
# Define your variables
output_data = {
    "start_date": start_date.isoformat(),
    "end_date": end_date.isoformat(),
    "bronze_adls": bronze_adls,
    "silver_adls": silver_adls,
    "gold_adls": gold_adls
}

dbutils.jobs.taskValues.set(key="bronze_output", value = output_data)